# Validación cruzada (k-fold) — BETO + LoRA

Separado en dos bloques por `weight_decay` (0.01 y 0.05), cada uno con
`r ∈ {8,16,32}` × `lr ∈ {1e-4,3e-4,1e-3}` = 9 combinaciones × 3 folds = 27
entrenamientos por bloque. Corré un bloque, y si querés parar ahí y seguir
otro día, el otro bloque es independiente. Al final se unen los dos CSVs.

In [ ]:
!git clone https://github.com/camistrika/BETO_HUMOR.git
%cd BETO_HUMOR
!pip install -e . -q

In [ ]:
!pip install -q torchao --upgrade
!pip install -q transformers peft datasets scikit-learn pyyaml

In [ ]:
%cd /content/BETO_HUMOR

In [ ]:
import os
import numpy as np
import pandas as pd
from itertools import product
from sklearn.model_selection import StratifiedKFold
from transformers import AutoTokenizer

from betohumor.config import DataConfig, BetoConfig, LoraConfig
from betohumor.utils import set_seed
from betohumor.dataset import load_and_split
from betohumor.models.lora import build_beto_lora
from betohumor.hyperparam_search import run_one

## 1. Datos y configuraciones (común a ambos bloques)

In [ ]:
data_config = DataConfig()
beto_config = BetoConfig()
set_seed(data_config.seed)

df_train, df_val, df_test = load_and_split(data_config)

# Unimos train+val para repartir en folds. 
df_full = pd.concat([df_train, df_val]).reset_index(drop=True)

tokenizer = AutoTokenizer.from_pretrained(beto_config.base_model)
label_col = data_config.label_col

LR_VALUES = [1e-4, 3e-4, 1e-3]
R_VALUES  = [8, 16, 32]
N_FOLDS   = 3

def builder(params):
    lora_config = LoraConfig(r=params['r'], lora_alpha=params['lora_alpha'])
    return build_beto_lora(beto_config, lora_config)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=data_config.seed)

def run_cv_block(weight_decay, csv_suffix):
    """Corre el CV completo para un solo valor de weight_decay y guarda
    los resultados en results/cv_results_lora_{csv_suffix}.csv"""
    params_grid = [
        {'r': r, 'lora_alpha': r * 2, 'learning_rate': lr, 'weight_decay': weight_decay}
        for lr, r in product(LR_VALUES, R_VALUES)
    ]

    all_fold_results = []
    for params in params_grid:
        fold_scores = []
        for fold_i, (train_idx, val_idx) in enumerate(skf.split(df_full, df_full[label_col])):
            run_name = f"r{params['r']}_lr{params['learning_rate']}_wd{params['weight_decay']}_fold{fold_i+1}"
            print(f"\n--- {run_name} ---")

            df_fold_train = df_full.iloc[train_idx].reset_index(drop=True)
            df_fold_val   = df_full.iloc[val_idx].reset_index(drop=True)

            output_dir = f"results/checkpoints/cv_lora/{run_name}"
            macro_f1, trainer = run_one(
                builder, params, beto_config, data_config,
                df_fold_train, df_fold_val, tokenizer, output_dir, seed=data_config.seed,
            )

            fold_scores.append(macro_f1)
            all_fold_results.append({**params, 'fold': fold_i + 1, 'macro_f1': macro_f1})
            print(f"Fold {fold_i+1} Macro F1: {macro_f1:.4f}")

        print(f"\n=== r={params['r']} lr={params['learning_rate']} wd={params['weight_decay']} -> "
              f"Media: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f} ===")

    os.makedirs('results', exist_ok=True)
    df_folds = pd.DataFrame(all_fold_results)
    df_folds.to_csv(f'results/cv_results_lora_{csv_suffix}.csv', index=False)
    print(f"\nGuardado en results/cv_results_lora_{csv_suffix}.csv")
    return df_folds

## 2. Bloque A — weight_decay = 0.01
9 combinaciones × 3 folds = 27 entrenamientos.

In [ ]:
df_folds_wd01 = run_cv_block(weight_decay=0.01, csv_suffix='wd01')

## 3. Bloque B — weight_decay = 0.05
9 combinaciones × 3 folds = 27 entrenamientos.

Independiente del bloque anterior.

In [ ]:
df_folds_wd05 = run_cv_block(weight_decay=0.05, csv_suffix='wd05')

## 4. Unir ambos resultados

Si los dos bloques se corrieron en la MISMA sesión, esta celda ya tiene
`df_folds_wd01` y `df_folds_wd05` en memoria. Si se corrieron en sesiones
DISTINTAS (por ejemplo, cada bloque un día diferente), comentá las líneas
de `run_cv_block` arriba y descomentá el bloque de `pd.read_csv` de abajo
para leer los CSVs ya guardados en vez de tenerlos en memoria.

In [ ]:

df_folds_wd01 = pd.read_csv('results/cv_results_lora_wd01.csv')
df_folds_wd05 = pd.read_csv('results/cv_results_lora_wd05.csv')

df_folds = pd.concat([df_folds_wd01, df_folds_wd05]).reset_index(drop=True)
df_folds.to_csv('results/cv_results_lora.csv', index=False)

df_summary = (
    df_folds
    .groupby(['r', 'lora_alpha', 'learning_rate', 'weight_decay'])['macro_f1']
    .agg(mean_macro_f1='mean', std_macro_f1='std')
    .reset_index()
    .sort_values('mean_macro_f1', ascending=False)
)
df_summary.to_csv('results/cv_results_lora_summary.csv', index=False)
df_summary